# Train `shared_convnext_stardist_decoder` (GS40 paths from `train_convnext_T224.py`)

This notebook pulls **dataset locations** from `convnextv2_tiny_224/train_convnext_T224.py`, writes a **YAML config** for the multitask trainer, and lists the **terminal / Jupyter** commands to run training.

**Important:** Your ConvNeXt CellViT pipeline uses **`train/labels/*.csv`** (centroids + class). The multitask StarDist-style trainer expects **`train/labels/{stem}.tif`** (or `.png`) **instance label images** (0 = background, each nucleus a unique integer id) and optional `{stem}_inst2class.json`. You must **rasterize** instances (e.g. from StarDist/GeoJSON) into those masks, or point `train_labels_dir` / `val_labels_dir` at a folder you already filled. The paths below match your GS40 **images** and **split CSVs**; label folders are set to sensible **sibling directories** you can populate.

## Canonical paths (copied from `train_convnext_T224.py`)

| Symbol | Path |
|--------|------|
| `DATASET_ROOT` | `\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced` |
| Images (all tiles) | `DATASET_ROOT\train\images` |
| Centroid CSV labels (CellViT) | `DATASET_ROOT\train\labels` |
| `label_map.yaml` | `DATASET_ROOT\label_map.yaml` |
| Fold 0 train list | `DATASET_ROOT\splits\fold_0\train.csv` |
| Fold 0 val list | `DATASET_ROOT\splits\fold_0\val.csv` |
| Original ConvNeXt run dir | `DATASET_ROOT\convnext_runs\run_fold0_convnextv2-tiny-22k-224` |

**Patch size** in the original script: **256** (matches `config.example.yaml`).

In [ ]:
from pathlib import Path
import shutil

# --- Same as convnextv2_tiny_224/train_convnext_T224.py ---
DATASET_ROOT = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced")
FOLD_DIR = DATASET_ROOT / "splits" / "fold_0"
TRAIN_IMAGES_DIR = DATASET_ROOT / "train" / "images"
TRAIN_LABELS_CSV_DIR = DATASET_ROOT / "train" / "labels"  # CellViT CSVs (not used directly by multitask train)
LABEL_MAP_YAML = DATASET_ROOT / "label_map.yaml"
TRAIN_SPLIT_CSV = FOLD_DIR / "train.csv"
VAL_SPLIT_CSV = FOLD_DIR / "val.csv"

# Subfolders for multitask training (you create instance masks under *_INSTANCE)
MULTITASK_PREP = DATASET_ROOT / "stardist_multitask_ready"
MT_TRAIN_IMAGES = MULTITASK_PREP / "train_images"
MT_VAL_IMAGES = MULTITASK_PREP / "val_images"
MT_TRAIN_INSTANCE_LABELS = MULTITASK_PREP / "train_instance_labels"
MT_VAL_INSTANCE_LABELS = MULTITASK_PREP / "val_instance_labels"

REPO_ROOT = Path(r"C:\Users\Andre\cursor_projects\Convnext_stardist")
CONFIG_OUT = REPO_ROOT / "shared_convnext_stardist_decoder" / "config_gs40_multitask.yaml"
CKPT_OUT = DATASET_ROOT / "convnext_stardist_multitask_runs" / "run_fold0"

for p in (MT_TRAIN_IMAGES, MT_VAL_IMAGES, MT_TRAIN_INSTANCE_LABELS, MT_VAL_INSTANCE_LABELS, CKPT_OUT):
    p.mkdir(parents=True, exist_ok=True)

print("DATASET_ROOT exists:", DATASET_ROOT.is_dir())
print("TRAIN_SPLIT_CSV:", TRAIN_SPLIT_CSV.is_file(), TRAIN_SPLIT_CSV)
print("VAL_SPLIT_CSV:  ", VAL_SPLIT_CSV.is_file(), VAL_SPLIT_CSV)
print("Config will be:", CONFIG_OUT)

## Optional: copy only fold_0 train/val **images** into `stardist_multitask_ready/`

Run the next cell once so `train.py` sees **non-overlapping** train vs val **image** folders. You still need matching **instance** `.tif` masks (same stems) under `train_instance_labels` / `val_instance_labels`.

In [ ]:
import pandas as pd

IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]


def read_split_stems(split_csv: Path) -> list[str]:
    df = pd.read_csv(split_csv)
    for col in ["id", "tile_id", "name", "filename", "file", "image"]:
        if col in df.columns:
            return [Path(v).stem for v in df[col].astype(str).tolist()]
    return [Path(v).stem for v in df.iloc[:, 0].astype(str).tolist()]


def find_image(stem: str, src_dir: Path) -> Path | None:
    for ext in IMAGE_EXTENSIONS:
        p = src_dir / f"{stem}{ext}"
        if p.is_file():
            return p
    return None


def copy_split_images(stems: list[str], dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for stem in stems:
        src = find_image(stem, TRAIN_IMAGES_DIR)
        if src is None:
            continue
        shutil.copy2(src, dest / src.name)
        n += 1
    return n


tr_stems = read_split_stems(TRAIN_SPLIT_CSV)
va_stems = read_split_stems(VAL_SPLIT_CSV)
print("train stems:", len(tr_stems), "val stems:", len(va_stems))
print("Copied train images:", copy_split_images(tr_stems, MT_TRAIN_IMAGES))
print("Copied val images:  ", copy_split_images(va_stems, MT_VAL_IMAGES))

In [ ]:
import yaml

# Class names aligned with train_convnext_T224 / GS40 label_map order is project-specific;
# keep the same 19 names as config.example.yaml — edit if your label_map differs.
cfg = {
    "experiment_name": "convnext_stardist_mt_gs40_fold0",
    "data": {
        "train_images_dir": str(MT_TRAIN_IMAGES),
        "train_labels_dir": str(MT_TRAIN_INSTANCE_LABELS),
        "val_images_dir": str(MT_VAL_IMAGES),
        "val_labels_dir": str(MT_VAL_INSTANCE_LABELS),
    },
    "model": {
        "backbone": "convnextv2_tiny.fcmae_ft_in22k_in1k_224",
        "pretrained": True,
        "n_rays": 32,
        "class_names": [
            "bladder", "bone", "brain", "collagen", "ear", "eye", "gi", "heart",
            "kidney", "liver", "lungs", "mesokidney", "nontissue", "pancreas",
            "skull", "spleen", "spleen2", "thymus", "thyroid",
        ],
        "num_classes": 19,
        "decoder_channels": 128,
    },
    "train": {
        "patch_size": 256,
        "batch_size": 4,
        "num_workers": 4,
        "epochs": 100,
        "lr": 1.0e-4,
        "weight_decay": 0.01,
        "amp": True,
        "freeze_backbone_epochs": 5,
        "loss_w_prob": 1.0,
        "loss_w_dist": 0.05,
        "loss_w_cls": 0.5,
        "log_every": 20,
    },
    "checkpoint": {
        "out_dir": str(CKPT_OUT),
        "save_every": 1,
    },
    "infer": {
        "prob_thresh": 0.45,
        "nms_dist": 3,
        "sample_offsets": [0, 128],
    },
}

CONFIG_OUT.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Wrote", CONFIG_OUT)

## Install dependencies (terminal once)

Install **PyTorch** for your CUDA version first, then:

```powershell
cd C:\Users\Andre\cursor_projects\Convnext_stardist
pip install -r shared_convnext_stardist_decoder\requirements.txt
```

## Train (PowerShell or **Anaconda Prompt**)

From the **repository root** (`Convnext_stardist`):

```powershell
cd C:\Users\Andre\cursor_projects\Convnext_stardist
py -3 -m shared_convnext_stardist_decoder.train --config shared_convnext_stardist_decoder\config_gs40_multitask.yaml
```

Weights and `idx2label.json` are written under:

`\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced\convnext_stardist_multitask_runs\run_fold0\`

## Train from inside Jupyter (same effect)

Run the next cell after opening this notebook with **kernel cwd = repo root** (or `%cd` there first).

In [ ]:
# Run training from the notebook (optional)
%cd C:\Users\Andre\cursor_projects\Convnext_stardist

!py -3 -m shared_convnext_stardist_decoder.train --config shared_convnext_stardist_decoder\config_gs40_multitask.yaml